# 4.5 Outlier Detection and Treatment

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.05_outlier-detection-and-treatment.ipynb)

Companion code for Section 4.5 of the book. Run the cells in order from top to bottom.

**Example 4.5.1**

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
# Generate data with injected outliers
n = 300
income = np.concatenate([
    np.random.normal(60000, 15000, n),   # Normal distribution
    [500000, 750000, 1200000, -5000]      # Injected outliers
])

# ── Method 1: Z-Score (assumes normal distribution) ──────────────
z_scores    = np.abs(stats.zscore(income))
z_outliers  = np.where(z_scores > 3)[0]
print(f'Z-score (|z|>3): {len(z_outliers)} outliers detected')
print(f'  Values: {income[z_outliers].astype(int).tolist()}')

# ── Method 2: IQR (Tukey Fences) - robust to skew ────────────────
Q1, Q3 = np.percentile(income, [25, 75])
IQR     = Q3 - Q1
lower   = Q1 - 1.5 * IQR
upper   = Q3 + 1.5 * IQR
iqr_mask    = (income < lower) | (income > upper)
print(f'\nIQR method: {iqr_mask.sum()} outliers detected')
print(f'  Fences: [{lower:,.0f}, {upper:,.0f}]')
print(f'  Values: {income[iqr_mask].astype(int).tolist()}')

# ── Method 3: Modified Z-Score (MAD) - most robust ───────────────
# Uses Median Absolute Deviation instead of std
median   = np.median(income)
mad      = np.median(np.abs(income - median))
mod_z    = 0.6745 * np.abs(income - median) / mad
mad_outliers = np.where(mod_z > 3.5)[0]
print(f'\nModified Z-score (MAD): {len(mad_outliers)} outliers detected')

# ── Method 4: Percentile clipping ────────────────────────────────
low_p, high_p = np.percentile(income, [1, 99])
income_clipped = np.clip(income, low_p, high_p)
print(f'\nPercentile clip [1st,99th]: [{low_p:,.0f}, {high_p:,.0f}]')
print(f'  Max after clip: {income_clipped.max():,.0f}')

**Example 4.5.2**

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import numpy as np

np.random.seed(42)
# Multivariate data with outliers - harder to detect univariately
n_normal   = 300
n_outliers = 20

X_normal   = np.random.normal([0, 0], [1, 1], (n_normal, 2))
X_outliers = np.random.uniform(-6, 6, (n_outliers, 2))  # Scattered far from centre
X = np.vstack([X_normal, X_outliers])
y_true = np.hstack([np.ones(n_normal), -np.ones(n_outliers)])  # 1=normal, -1=outlier

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Isolation Forest ─────────────────────────────────────────────
# Outliers are isolated in fewer splits than normal points
iso = IsolationForest(contamination=0.06, random_state=42)
iso_pred = iso.fit_predict(X_scaled)   # 1=inlier, -1=outlier
iso_acc  = (iso_pred == y_true).mean()
print(f'Isolation Forest: {(iso_pred==-1).sum()} outliers detected, accuracy={iso_acc:.1%}')

# ── Local Outlier Factor ─────────────────────────────────────────
# Compares local density of a point to its neighbours
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
lof_pred = lof.fit_predict(X_scaled)
lof_acc  = (lof_pred == y_true).mean()
print(f'Local Outlier Factor: {(lof_pred==-1).sum()} outliers detected, accuracy={lof_acc:.1%}')

# ── One-Class SVM ────────────────────────────────────────────────
oc_svm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.06)
svm_pred = oc_svm.fit_predict(X_scaled)
svm_acc  = (svm_pred == y_true).mean()
print(f'One-Class SVM: {(svm_pred==-1).sum()} outliers detected, accuracy={svm_acc:.1%}')

# ── Outlier Treatment Options ─────────────────────────────────────
print('\nOutlier treatment options:')
print('1. Remove:    X_clean = X[iso_pred == 1]           # Drop outlier rows')
print('2. Cap:       np.clip(X, Q1-1.5*IQR, Q3+1.5*IQR) # Winsorise')
print('3. Transform: np.log1p(X)                          # Reduce skew')
print('4. Flag:      df["is_outlier"] = (iso_pred == -1)  # Keep but flag')
print('5. Ignore:    Use robust models (trees, SVMs)      # Not all models are sensitive')